# Main results and subset ranks

CPU only. Requires the saved five-seed two-family selection results for both models. This notebook summarizes existing results; it does not rerun selection.

In [ ]:
from pathlib import Path
import os
import json
from IPython.display import display

# Edit this ONE setting if your collections live elsewhere.
DATA_ROOT = Path(os.environ.get("B3IT_DATA_ROOT", "/content/drive/MyDrive"))
if not DATA_ROOT.exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        raise FileNotFoundError("Set B3IT_DATA_ROOT to the directory containing both model folders.")
assert DATA_ROOT.exists(), DATA_ROOT
collections = {
    "Qwen": DATA_ROOT / "B3IT_Qwen05_Noise_Pruning" / "expanded_scales_1000samples_v1",
    "Phi": DATA_ROOT / "B3IT_Phi4mini_Noise_Pruning" / "expanded_scales_1000samples_v1",
}
folders = {model: folder / "extension_selection_repeated_v1" for model, folder in collections.items()}


## Report table calculated from CSV files

In [ ]:
import pandas as pd
from IPython.display import display

tables = []

for model, folder in collections.items():
    source = (
        folder / "extension_selection_repeated_v1"
        / "all_seeds_comparison.csv"
    )
    data = pd.read_csv(source)

    assert len(data) == 15
    assert not data.duplicated(["seed", "held_out_type"]).any()
    assert data.groupby("held_out_type").size().eq(5).all()

    grouped = data.groupby("held_out_type", sort=False)

    table = grouped[
        ["performance_auc", "entropy_auc", "random_mean_auc"]
    ].agg(["mean", "std"])

    formatted = pd.DataFrame(index=table.index)

    for column, label in [
        ("performance_auc", "Performance-based"),
        ("entropy_auc", "Entropy-based"),
        ("random_mean_auc", "Uniform random"),
    ]:
        formatted[label] = [
            f"{mean:.4f} ± {sd:.4f}"
            for mean, sd in zip(
                table[(column, "mean")],
                table[(column, "std")],
            )
        ]

    formatted.insert(0, "Model", model)
    tables.append(formatted.reset_index())

report_table = pd.concat(tables, ignore_index=True).rename(
    columns={"held_out_type": "Held-out type"}
)

display(report_table)

## Ranks, oracle gaps, and seed 303 diagnostic
The oracle is retrospective and optimistic. Seed SD is descriptive, not a confidence interval.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

rows = []

for model, folder in folders.items():
    for path in sorted(folder.glob("seed_*.json")):
        saved = json.loads(path.read_text())
        subsets = pd.DataFrame(saved["subset_results"])

        for held_out, selection in saved["selections"].items():
            group = subsets[
                subsets["held_out_type"] == held_out
            ].copy()

            assert len(group) == 120
            assert group["prompts"].map(
                lambda p: tuple(sorted(p))
            ).nunique() == 120

            aucs = group["auc"].to_numpy(dtype=float)
            best_auc = aucs.max()

            for strategy, field in [
                ("Performance", "performance_prompts"),
                ("Entropy", "entropy_prompts"),
            ]:
                selected = frozenset(selection[field])
                match = group["prompts"].map(
                    lambda prompts: frozenset(prompts) == selected
                )

                assert match.sum() == 1, (
                    model, saved["seed"], held_out, strategy
                )

                selected_auc = float(group.loc[match, "auc"].iloc[0])

                # Tolerance only handles floating-point equality.
                tied = np.isclose(
                    aucs, selected_auc, rtol=0, atol=1e-12
                )
                better = (aucs > selected_auc) & ~tied

                rows.append({
                    "model": model,
                    "seed": saved["seed"],
                    "held_out_type": held_out,
                    "strategy": strategy,
                    "selected_auc": selected_auc,
                    "rank_of_120": 1 + int(better.sum()),
                    "tied_subsets": int(tied.sum()),
                    "best_subset_auc": best_auc,
                    "gap_to_best": best_auc - selected_auc,
                    "random_mean_auc": aucs.mean(),
                    "subset_sd": aucs.std(ddof=1),
                    "selected_prompts": json.dumps(
                        selection[field], ensure_ascii=False
                    ),
                })

ranks = pd.DataFrame(rows)

summary = (
    ranks.groupby(
        ["model", "held_out_type", "strategy"], sort=False
    )
    .agg(
        mean_auc=("selected_auc", "mean"),
        median_rank=("rank_of_120", "median"),
        best_rank=("rank_of_120", "min"),
        worst_rank=("rank_of_120", "max"),
        mean_oracle_auc=("best_subset_auc", "mean"),
        mean_gap_to_best=("gap_to_best", "mean"),
        mean_subset_sd=("subset_sd", "mean"),
    )
    .reset_index()
)

for model, folder in folders.items():
    ranks[ranks["model"] == model].to_csv(
        folder / "selection_subset_ranks.csv", index=False
    )
    summary[summary["model"] == model].to_csv(
        folder / "selection_subset_rank_summary.csv", index=False
    )

print("RÉSUMÉ SUR LES CINQ SEEDS")
display(summary.round(4))

print("\nPHI : RANDOM PRUNING — VÉRIFICATION DE LA SEED 303")
display(
    ranks[
        (ranks["model"] == "Phi")
        & (ranks["held_out_type"] == "random")
    ][[
        "seed", "strategy", "selected_auc",
        "rank_of_120", "gap_to_best", "selected_prompts"
    ]].round(4)
)

print("\nTableaux sauvegardés dans les deux dossiers d’analyse.")